## Omni Recognition with Qwen3-VL

By improving the quality and diversity of pre-training data, the model can now recognize a much wider range of objects — from celebrities, anime characters, products, and landmarks, to animals and plants — covering both everyday life and professional “recognize anything” needs.

This notebook demonstrates how to use Qwen3-VL for omni recognition. It takes an image and a query, and then uses the model to interpret the user's query on the image. 

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
# Required: OPENAI_API_KEY, OPENAI_BASE_URL, MODEL_ID
load_dotenv()

# Set OpenAI API credentials (required for OpenAI-compatible server)
API_KEY = os.environ["OPENAI_API_KEY"]
BASE_URL = os.environ["OPENAI_BASE_URL"]
MODEL_ID = os.environ["MODEL_ID"]

print("✓ Environment variables loaded successfully")
print(f"  API_KEY: {API_KEY[:8]}..." if API_KEY else "  API_KEY: (empty)")
print(f"  BASE_URL: {BASE_URL}")
print(f"  MODEL_ID: {MODEL_ID}")


## \[Setup\]

In [ ]:
%pip install git+https://github.com/huggingface/transformers
%pip install qwen-vl-utils
%pip install qwen_agent
%pip install openai

## Load plotting and inference util.

In [ ]:
import json
import random
from PIL import Image, ImageDraw, ImageFont
from openai import OpenAI
import os
import base64
import requests
import copy
import traceback
import time


def inference_with_openai_api(img_url, prompt, min_pixels=64 * 32 * 32, max_pixels=9800* 32 * 32):
    import base64
    import os
    if os.path.exists(img_url):
        with open(img_url, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    elif img_url.startswith("http://") or img_url.startswith("https://"):
        response = requests.get(img_url)
        response.raise_for_status()
        base64_image = base64.b64encode(response.content).decode("utf-8")
    else:
        raise ValueError("Invalid image URL")
    client = OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    },
                    "min_pixels": min_pixels,
                    "max_pixels": max_pixels
                },
                {"type": "text", "text": prompt},
            ],
        }
    ]
    completion = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
    )
    return completion.choices[0].message.content    


#### 1. Object Recognition

In [ ]:
image_path = "assets/omni_recognition/sample-celebrity-2.jpg"
prompt = "Who is this"

image = Image.open(image_path)
image.thumbnail([640,640], Image.Resampling.LANCZOS)
display(image)

response = inference_with_openai_api(image_path, prompt)
print(response)

#### 2 Object Spotting

With this update, we can now identify multiple objects in an image and find their coordinates.
Note that we use relative coordinates, and the coordinates are scaled from 0 to 1000.

In [ ]:
image_path = "./assets/omni_recognition/sample-food.jpeg"
prompt = "Identify food in the image and return their bounding box and Chinese and English name in JSON format."

image = Image.open(image_path)
image.thumbnail([640,640], Image.Resampling.LANCZOS)
display(image)

response = inference_with_openai_api(image_path, prompt)
print(response)


In [ ]:
image_path = "./assets/omni_recognition/sample-anime.jpeg"
prompt = "Who are the anime characters in the image? Please show the bounding boxes of all characters and their names in Chinese and English in JSON format."

image = Image.open(image_path)
image.thumbnail([640,640], Image.Resampling.LANCZOS)
display(image)

response = inference_with_openai_api(image_path, prompt)
print(response)